# 이미지를 텍스트로 검색/생성 RAG

In [ ]:
%pip install -Uq langchain langchain-openai langchain-chroma

In [ ]:
# 구글드라이브 연동
from google.colab import drive
drive.mount('/content/drive')  # 내 구글드라이브를 /content/drive 경로로 마운트

BASE_PATH = '/content/drive/MyDrive/SK네트웍스 Family AI 캠프 23기/05_multimodal_rag'  # 기본작업경로 변수

In [ ]:
import os
from google.colab import userdata  # Colab 비밀키 저장소 접근 모듈

# 환경변수 설정
os.environ['LANGSMITH_TRACING'] = 'true'  # LangSmith 추적 기능 활성화
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_API_KEY'] = userdata.get('LANGSMITH_API_KEY')
os.environ['LANGSMITH_PROJECT'] = 'skn23-multimodal-rag'
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')


## 이미지 캡셔닝 & 벡터 DB 저장

In [ ]:
import base64  # 바이너리 데이터를 Base64 문자열로 인코딩 모듈
from PIL import Image  # 이미지 처리

# 이미지 파일을 Base64 문자열로 변환하는 함수
def image_to_base64(image_path):
    with open(image_path, 'rb') as f:  # 이미지를 바이너리 읽기 모드로 열기
      encoded = base64.b64encode(f.read()).decode('utf-8')  # Base64 인코딩 후 문자열로 변환
    return encoded  # 인코딩된 문자열 반환

- 이렇게 인코딩된 문자열은 json의 값으로 보내면 이미지를 보낼때 매우 편하게 보낼 수 있다.

In [ ]:
from openai import OpenAI

client = OpenAI()  # OpenAI 클라이언트 객체

# 이미지 파일을 받아 안전교육 및 화재대피 관점에서 요약하는 함수
def summarize_image(image_path):
  encoded = image_to_base64(image_path)

  # Chat Completions API 호출
  response = client.chat.completions.create(
    model = 'gpt-4.1-mini',
    messages = [
        {
            'role': 'system',
            'content': '''
아래 아미지를 안전교육 및 화재대피 관점에서
주제문구와 각 장면에 대한 정보를 빠짐없이 한국어로 설명해주세요.
'''  # 모델에게 부여하는 지침(역할 및 설명 방식 정의)
        },
        {
            'role': 'user',
            'content': [{
                'type': 'image_url',  # 이미지 입력 타입 지정
                'image_url': {
                    'url': f'data:image/jpeg;base64,{encoded}'  # Base64 인코딩된 이미지 전달
                }
            }]
        }
    ],
    max_tokens = 1024  # 최대 생성 토큰 수
  )
  return response.choices[0].message.content  # 모델이 생성한 텍스트 응답

print(summarize_image(f'{BASE_PATH}/figures/figure-1.jpg'))


In [ ]:
from langchain_chroma.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from tqdm.auto import tqdm
import os

persistent_dir = f'{BASE_PATH}/chroma'  # 벡터 DB 저장 경로
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')  # OpenAI 임베딩 모델 객체

vector_db = Chroma(
    collection_name = 'multimodal_rag',  # 컬렉션(테이블) 이름
    embedding_function = embeddings,     # 임베딩 함수 연결
    persist_directory = persistent_dir   # 디스크 저장 경로
)

In [ ]:
figure_dir = os.path.join(BASE_PATH, 'figures')  # 이미지 폴더 경로

image_files = [
    os.path.join(figure_dir, filename)  # 전체 경로 저장
      for filename in os.listdir(figure_dir)  # 폴더 내 파일 목록 순회
        if filename.lower().endswith('.jpg')  # 확장자가 .jpg 파일
]

for img_path in tqdm(image_files):
  b64_str = image_to_base64(img_path)  # 이미지 Base64 인코딩
  caption = summarize_image(img_path)  # 이미지 요약 생성
  vector_db.add_texts(
      texts = [caption],  # 벡터화해서 저장
      metadatas = [{  # 메타데이터 저장
          'base64_image': b64_str,
          'image_path': img_path
      }]
  )

In [ ]:
docs = vector_db._collection.get()  # 컬렉션에 저장된 전체 문서 및 메타데이터

# 문서와 메타데이터를 함께 순회하면서 출력
for i, (page_content, metadata) in enumerate(zip(docs['documents'], docs['metadatas'])):
  print(f'[{i + 1}]')
  print(f'{page_content = }')
  print(f'{metadata = }')
  print()

In [ ]:
from io import BytesIO  # 바이트 데이터를 메모리 파일처럼 다루기 위한 모듈
from PIL import Image

# Base64 문자열 -> 이미지로 화면 출력하는 함수
def show_base64_image(b64_str):
  img = Image.open(BytesIO(base64.b64decode(b64_str)))  # Base64 디코딩 후 이미지 객체 생성
  display(img)

show_base64_image(docs['metadatas'][0]['base64_image'])

In [ ]:
show_base64_image(docs['metadatas'][2]['base64_image'])

In [ ]:
show_base64_image(docs['metadatas'][3]['base64_image'])

In [ ]:
# 벡터 DB 유사도 검색 후 이미지 및 캡션 출력
query = '소화기 어떻게 써야돼?'
docs = vector_db.similarity_search(query, k=2)  # 질문과 유사한 상위 2개 문서 검색

for i, doc in enumerate(docs):
  print(f'[{i + 1}]\n {doc.page_content}')  # 해당 문서의 캡션(요약 내용)
  show_base64_image(doc.metadata['base64_image']) # 메타데이터에 저장된 이미지 출력

## Multimodal RAG 구현
- agentic rag 구현 (middleware 기반 검색된 문서 제공)
- text + image 응답

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, AgentState  # 에이전트 미들웨어 / 상태 타입
from langchain_core.documents import Document  # 검색 결과 문서 타입
from typing import Any  # 타입 힌트용

# 에이전트가 유지할 상태 스키마 정의 (검색 문서 컨텍스트 저장)
class State(AgentState):
  context: list[Document]

# 모델 호출 전 문서 검색 및 프롬프트 재작성하는 미들웨어
class RetrieveDocumentsMiddleware(AgentMiddleware[State]):
  state_schema = State  # 이 미들웨어가 사용하는 상태 스키마

  def before_model(self, state: AgentState) -> dict[str, Any] | None:
    last_message = state['messages'][-1]  # 마지막 사용자 메시지
    retrieved_docs = vector_db.similarity_search(last_message.text, k=2)  # 사용자 메시지 질문과 유사한 문서 2개 검색

    docs_content = '\n\n'.join(doc.page_content for doc in retrieved_docs)  # 문서 본문을 하나의 문자열로 결합

    prompt = f"""
제공된 '참조 문서'의 내용을 바탕으로 '사용자 질문'에 대해 상세히 답변하라.

제약 조건:
- 답변은 반드시 제공된 문서 내의 정보만을 근거로 작성한다.
- 문서에서 답을 찾을 수 없는 경우, "제공된 정보에서는 해당 내용을 확인할 수 없습니다"라고 답변한다.
- 논리적이고 가독성이 좋게 구조화하여 설명한다.

참조 문서:
{docs_content}

사용자 질문:
{last_message.text}
"""
    return {
        "messages" : [last_message.model_copy(update={'context': prompt})],  # 사용자 메시지를 프롬프트로 바꿔 모델에 전달
        "context" : retrieved_docs  # 검색된 결과는 context에 저장
    }

llm = init_chat_model('openai:gpt-4.1-mini')
agent = create_agent(
    llm,
    tools = [],
    middleware = [RetrieveDocumentsMiddleware()]  # 모델 호출 전 검색 / 프롬프트 주입 미들웨어
)

response = agent.invoke({
    'messages' : [
      ('human', '화재시 대피 요령은 어떻게 돼?')
    ]
})  # 에이전트 실행 (미들웨어가 검색 후 프롬프트 변경하고 모델 호출)

print(response['messages'][-1].content)  # 최종 모델 응답

print('[검색된 내용]')
for doc in response['context']:  # 검색된 문서들을 순회
  show_base64_image(doc.metadata['base64_image'])  # 이미지 출력
  print(doc.page_content)  # 각 문서의 캡션(요약 텍스트)
  print('---------' * 100)